# K7 · Prepare demo papers for the website
Processes a few papers with the full pipeline on a Kaggle GPU (fast), writes the Gemini overview, and uploads each
processed paper to your private results dataset (`demo_papers/`). On the website they then appear under
**"Try a demo paper (instant)"** and open in seconds.

Needs: **GPU T4 x2** (or P100), **Internet on**, secrets **`HF_TOKEN`** (write) and **`GEMINI_API_KEY`**. About 10 minutes.

## 0. Setup (Kaggle)
Before running: right panel → **Accelerator: GPU T4 x2 (or P100)**, **Internet: On** (needs a phone-verified Kaggle account),
and **Add-ons → Secrets → add `HF_TOKEN`** (your Hugging Face write token) and tick it for this notebook.

Kaggle has no Google Drive: results go to `/kaggle/working/ResearchLens` **and** every model + its results are pushed to your
private Hugging Face account. The last cell also backs up all results to a private HF dataset.

In [ ]:
import os, sys, subprocess
REPO_URL = "https://github.com/mkavinvelavan/ResearchLens.git"
DRIVE_ROOT = "/kaggle/working/ResearchLens"          # results (Kaggle's saved output folder)
SCRATCH = "/tmp/rl"                                    # data, checkpoints, final models (large, not saved)

os.environ["CUDA_VISIBLE_DEVICES"] = "0"               # one GPU, same setup as the Colab runs
for d in (DRIVE_ROOT, f"{SCRATCH}/content"):
    os.makedirs(d, exist_ok=True)
if not os.path.exists("/content"):                     # configs use /content/... paths (Colab layout)
    try:
        os.symlink(f"{SCRATCH}/content", "/content")
    except OSError as e:
        print("note: could not create /content link:", e)
os.makedirs("/content/data", exist_ok=True)

if not os.path.exists(f"{SCRATCH}/repo"):
    !git clone -q {REPO_URL} {SCRATCH}/repo
else:
    !git -C {SCRATCH}/repo pull -q
%cd {SCRATCH}/repo
!pip install -q -r requirements-colab.txt

os.environ.update({
    "RL_DRIVE_ROOT": DRIVE_ROOT, "RL_DATA_DIR": "/content/data",
    "RL_CKPT_DIR": f"{SCRATCH}/checkpoints", "RL_MODEL_DIR": f"{SCRATCH}/final_models",
    "TOKENIZERS_PARALLELISM": "false", "TQDM_DISABLE": "1",
})
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN found")
except Exception as e:
    print("WARNING: HF_TOKEN secret missing / not attached -> models will NOT be pushed to the Hub.", e)
!nvidia-smi --query-gpu=name,memory.total --format=csv

def run(module, config, run_name, seed, extra=""):
    # Train one (run, seed) unless it already finished; streams the log and stops on failure.
    import yaml
    tag = yaml.safe_load(open(config))["model_tag"]
    if os.path.exists(f"{DRIVE_ROOT}/results/{tag}/{run_name}/seed_{seed}/metrics.json"):
        print(f"skip {run_name} seed {seed} (finished)"); return
    if "--push_to_hub" in extra and not os.environ.get("HF_TOKEN"):
        extra = extra.replace("--push_to_hub", "")
    cmd = f"{sys.executable} -m {module} --config {config} --run_name {run_name} --seed {seed} {extra}"
    print(">>>", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    tail = []
    for line in p.stdout:
        print(line, end="", flush=True)
        tail = (tail + [line])[-40:]
    if p.wait() != 0:
        raise RuntimeError(f"Training failed: {run_name} seed {seed}. Last lines:\n" + "".join(tail))

## 1. Install the app's packages and load the Gemini key

In [ ]:
!pip install -q -r requirements-app.txt
from kaggle_secrets import UserSecretsClient
os.environ["GEMINI_API_KEY"] = UserSecretsClient().get_secret("GEMINI_API_KEY"); print("GEMINI_API_KEY loaded")

## 2. Demo papers
Swin Transformer and ViT (both vision transformers, a good pair for **Compare** and the **Gap Map**) and SciBERT
(an NLP paper). Downloaded from arXiv. To use other papers, add PDFs with **Add input → Upload** and list them here.

In [ ]:
import urllib.request, glob
PAPERS = "/kaggle/working/demo_pdfs"; os.makedirs(PAPERS, exist_ok=True)
DEMOS = [("swin_transformer.pdf", "https://arxiv.org/pdf/2103.14030"),
         ("vit.pdf", "https://arxiv.org/pdf/2010.11929"),
         ("scibert.pdf", "https://arxiv.org/pdf/1903.10676")]
for name, url in DEMOS:
    if not os.path.exists(f"{PAPERS}/{name}"):
        req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0 (ResearchLens course project)"})
        open(f"{PAPERS}/{name}", "wb").write(urllib.request.urlopen(req, timeout=60).read())
pdfs = sorted(glob.glob(f"{PAPERS}/*.pdf")); print(pdfs)

## 3. Process, write overviews, upload to Hugging Face

In [ ]:
from researchlens.pipeline.make_demo import make_demo
index = make_demo(pdfs, "/kaggle/working/demo_work")
for d in index:
    print(d)